### Review Bellman

Bellman turns a **long-term decision problem** into a sequence of **local decision problems**.

Bellman needs:

#### Recursive Objective
  
#### Markov Sufficiency

$$
P(s_{t+1}\mid s_t,a_t,\text{history})
=P(s_{t+1}\mid s_t,a_t)
$$

Ex 1.
>If I know your full state, you told me what you did 10 years ago, won't change next step prediction.

Ex 2. 
>state = having free time today. Action space is: dota or study.
>
>Can state determine action?
>
>It lacks key info: is there test tomorrow?

History is important, but:
>key info needs to be compressed into State 


<br>

#### Principle of Optimality


  >If an entire path is optimal, then the remaining path from any intermediate point must also be optimal.
  
  Example: Suppose the best route from New York to Los Angeles passes through Chicago.

  Then the route from Chicago to Los Angeles must also be the best possible route.

  Otherwise, you could replace that portion with a better route and improve the entire journey.


<br>

#### Time Consistency

  >A decision you consider optimal for your future self today should still be considered optimal when that future moment arrives.
  
  Example: On Sunday, you decide to wake up at 6 AM on Monday to exercise because long-term health matters most.

  Monday morning, you value an extra hour of sleep more. That's an example of time inconsistency.

---

### Question: Unknown Environment Model

In Dynamic Programming, we assume that the environment model is known.

Knowing the **state space** is not the same as knowing the **transition probabilities**.

- State space: What states are possible?
- Transition model: Given a state and an action, how likely is each next state?
- Reward rule: What reward is assigned to an outcome?
- Expected reward: What reward do we expect on average, given the probabilities of possible outcomes?

Even if the reward rule is known, the expected reward may be unknown because the transition probabilities are unknown.

### The Key Problem

We know that:
>repeatedly applying the Bellman operator -> converges

But applying Bellman operator requires:
>computing an **expectation** over possible transitions.

#### What if we don't know the environment model? 

therefore we can no longer compute that expectation directly.

---

### deterministic iteration -> stochastic approximation

Can we learn the value function directly from experience, without knowing the environment model?

if we expand Bellman:

$$
\begin{aligned}
V^\pi(s_t)
&=\mathbb E[r_{t+1}+\gamma V^\pi(s_{t+1})\mid s_t]\\
&=\mathbb E[r_{t+1}+\gamma r_{t+2}+\gamma^2V^\pi(s_{t+2})\mid s_t]\\
&=\cdots\\
&=\mathbb E\left[\sum_{k=0}^{\infty}\gamma^kr_{t+k+1}\mid s_t\right]
\end{aligned}
$$

- Approach 1: Monte Carlo Learning
  Bellman expanded all the way. Repeatedly experiment to approximate this value for each state.

- Approach 2: Temporal Difference Learning
  Bellman expands 1 step.

They can viewed as:

$$
\text{n-step TD}\longrightarrow\text{MC}
$$

#### Convergence Proof:

MC: based on the **Law of Large Numbers**, rather than Bellman iteration.

TD: based on stochastic approximation and the mathematical structure of the Bellman operator.

- The expected TD update follows the **Bellman operator**, which **converges** to fixed point.
- But it's expected value, not 1 time real world samlping. 
- To deal with that, stochastic approximation controls the effect of sampling noise, ensuring convergence under suitable conditions.

a.k.a
- Why does it move toward the correct value? → Bellman Contraction.
- Why doesn't sampling noise ruin convergence? → Stochastic Approximation.

$$
V(s_t)\leftarrow V(s_t)+\alpha\underbrace{\left[r_{t+1}+\gamma V(s_{t+1})-V(s_t)\right]}_{\text{TD Error}}
$$

TD target:

$$
y_t=r_{t+1}+\gamma V(s_{t+1})
$$

TD can be rewrite as:
$$
V(s_t)\leftarrow V(s_t)+\alpha(y_t-V(s_t))
$$

---

### Summary


| Method                                 | Goal 1: Policy Evaluation    | Goal 2: Control                    |
| -------------------------------------- | ---------------------------- | ---------------------------------- |
| Mathematical Foundation                | Bellman Expectation Equation | Bellman Optimality Equation        |
| Dynamic Programming (DP) — Model-Based | Iterative Policy Evaluation  | Policy Iteration / Value Iteration |
| Monte Carlo (MC) — Model-Free          | MC Prediction                | MC Control                         |
| Temporal Difference (TD) — Model-Free  | TD Prediction                | SARSA / Q-learning                 |


Goal 1: Policy Evaluation 
> given a policy, what is the value function (a.k.a evaluate its value)?

Goal 2: Control
> How to find the optimal policy? Dumbest way: enumerate through all policies then pick the best
> 
> term "control" originaly comes control theory

---
### Q-learning vs. SARSA (state, action, reward, state, action)

Just 1 line TD change! (there is another execution order change, see later)

```python
# Q-learning
target = r + gamma * np.max(Q[s_next])

# SARSA
target = r + gamma * Q[s_next, a_next]
```

Comparesion:

| Feature                        | SARSA                   | Q-learning                   |
| ------------------------------ | ----------------------- | ---------------------------- |
| Exploration strategy           | ε-greedy                | ε-greedy                     |
| Target policy                  | ε-greedy                | greedy                       |
| Updates Q-table?               | Yes                     | Yes                          |
| Automatically improves policy? | Yes                     | Yes                          |
| TD target                      | \\(r+\gamma Q(s',a')\\) | \\(r+\gamma\max_a Q(s',a)\\) |
| Learning type                  | On-policy               | Off-policy                   |


#### Q-learning:

idea of Bellman Optimality + TD + **off-policy(ε-greedy)**. 

The converge math proof is not easy, it's a different proof than TD learning math proof.

Why it's hard: It uses TD, but TD prediction is linear, Q-learning involves doing **max()**. So it needs to solves thew new issues and the issues facing TD converge proof, like contraction, noise, Asynchronous updates.

Proof skipped.


#### Why Is SARSA Safer Than Q-learning in Cliff Walking?

The key difference is their TD targets.

Suppose an agent reaches state $s'$, where:

| Action | Q-value |
|---|---|
| Safe | 0 |
| Cliff | -100 |

Due to ε-greedy exploration, the agent accidentally selects **Cliff** as its next action \(a'\). Assume $r=0$ and $\gamma=1$.

**SARSA:**

$$
r+\gamma Q(s',a')=0+(-100)=-100
$$

**Q-learning:**

$$
r+\gamma\max_a Q(s',a)=0+\max(0,-100)=0
$$

**Key insight:** Both algorithms know that falling off the cliff is bad. However:

- **SARSA** uses the action actually selected, so exploration mistakes affect the Q-values of earlier states.
- **Q-learning** always uses the best possible next action, ignoring exploration mistakes in its TD target.

As a result, SARSA learns to avoid risky paths near the cliff, while Q-learning tends to prefer the shortest path, even if it runs close to the cliff.

---

#### SARSA

see 5 things needed here?

$$
s \xrightarrow{a,r} s' \xrightarrow[\text{selected}]{a'} ?
$$

- Execute $a$, transitioning from $s$ to $s'$ and receiving reward $r$.
- At $s'$, select $a'$ using ε-greedy, but **do not execute it yet**.
- Update $Q(s,a)$ using the target $r + \gamma Q(s',a')$.
- Then execute $a'$ and proceed to the next iteration.

#### Q-learning

only 4

$$
s \xrightarrow{a,r} s'
$$

- Execute $a$, transitioning from $s$ to $s'$ and receiving reward $r$.
- Directly compute $\max_{a'} Q(s',a')$ without actually selecting $a'$.
- Update $Q(s,a)$ using the target $r + \gamma \max_{a'} Q(s',a')$.
- Then use ε-greedy to select the next action to execute.